## Step 1: Load Order Intake Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [2]:
FILE_PATH = "/Users/apple/AI Matics/Fibro/ROL Project/data/ROL Working.xlsx"

df = pd.read_excel(FILE_PATH, sheet_name="Data")

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

display(df.head())

Rows    : 4,817
Columns : 17


,OA Date,Item Code,Sum of Sales_Qty,Year,Week No,Week,Week-Year,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,2025-01-02,4960.85.125.250,4,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,4960.85.048.150,6,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-22,NaN,Week 10-2025
2,2025-01-02,4960.85.075.150,2,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-23,NaN,Week 10-2026
3,2025-01-02,4960.85.100.150,1,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-24,NaN,Week 11-2025
4,2025-01-02,4960.85.048.100,2,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,18.0,NaN,NaN,2024-25,NaN,Week 11-2026


## Step 2: Data Preparation

In [3]:
df["OA Date"] = pd.to_datetime(df["OA Date"])

df["Year"] = df["OA Date"].dt.isocalendar().year.astype(int)
df["Week"] = df["OA Date"].dt.isocalendar().week.astype(int)

df = df.sort_values("OA Date").reset_index(drop=True)

display(df.head())

,OA Date,Item Code,Sum of Sales_Qty,Year,Week No,Week,Week-Year,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,2025-01-02,4960.85.125.250,4,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,4960.85.048.150,6,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-22,NaN,Week 10-2025
2,2025-01-02,4960.85.075.150,2,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-23,NaN,Week 10-2026
3,2025-01-02,4960.85.100.150,1,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-24,NaN,Week 11-2025
4,2025-01-02,4960.85.048.100,2,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,18.0,NaN,NaN,2024-25,NaN,Week 11-2026


## Step 3: Aggregate Weekly Demand

In [4]:
weekly = (
    df.groupby(
        ["Item Code", "Year", "Week"],
        as_index=False
    )["Sum of Sales_Qty"]
    .sum()
)

weekly.rename(
    columns={"Sum of Sales_Qty": "Weekly Demand"},
    inplace=True
)

display(weekly.head())

,Item Code,Year,Week,Weekly Demand
0,4960.85.028.075,2025,6,2
1,4960.85.028.075,2025,23,2
2,4960.85.028.075,2025,29,48
3,4960.85.028.075,2025,31,4
4,4960.85.028.075,2025,32,2


## Step 4: Generate Product Summary

In [5]:
summary = (
    weekly.groupby("Item Code", as_index=False)
    .agg(
        Total_Sales=("Weekly Demand", "sum"),
        Average_Weekly_Demand=("Weekly Demand", "mean"),
        Maximum_Weekly_Demand=("Weekly Demand", "max"),
        Number_of_Weeks=("Weekly Demand", "count")
    )
)

display(summary.head())

,Item Code,Total_Sales,Average_Weekly_Demand,Maximum_Weekly_Demand,Number_of_Weeks
0,4960.85.028.075,231,8.555556,48,27
1,4960.85.028.100,24,3.428571,6,7
2,4960.85.028.150,42,8.400000,23,5
3,4960.85.038.050,2,2.000000,2,1
4,4960.85.038.075,377,10.189189,43,37


## Step 5: Apply Static Volume Classification

In [6]:
def volume_logic(total_sales):
    if total_sales <= 300:
        return 0
    elif total_sales <= 600:
        return 12
    else:
        return 24

summary["Volume Logic"] = summary["Total_Sales"].apply(volume_logic)

display(summary.head())

,Item Code,Total_Sales,Average_Weekly_Demand,Maximum_Weekly_Demand,Number_of_Weeks,Volume Logic
0,4960.85.028.075,231,8.555556,48,27,0
1,4960.85.028.100,24,3.428571,6,7,0
2,4960.85.028.150,42,8.400000,23,5,0
3,4960.85.038.050,2,2.000000,2,1,0
4,4960.85.038.075,377,10.189189,43,37,12


## Step 6: Select Product for ROL Calculation

In [7]:
# item_code = "4960.85.150.150"
# item_code = "4960.85.075.125"
# item_code = "4960.85.058.100"
# item_code = "4960.85.125.125"
item_code = "4960.85.075.100"

item_summary = summary.loc[
    summary["Item Code"] == item_code
]

display(item_summary)

,Item Code,Total_Sales,Average_Weekly_Demand,Maximum_Weekly_Demand,Number_of_Weeks,Volume Logic
20,4960.85.075.100,3320,46.111111,248,72,24


## Step 7: Extract Weekly Demand

In [8]:
weekly_item = (
    weekly.loc[
        weekly["Item Code"] == item_code,
        ["Year", "Week", "Weekly Demand"]
    ]
    .copy()
)

display(weekly_item)

,Year,Week,Weekly Demand
575,2025,2,54
576,2025,3,48
577,2025,4,28
578,2025,5,4
579,2025,6,69
...,...,...,...
642,2026,18,71
643,2026,19,25
644,2026,20,24
645,2026,21,46


In [9]:
bin_size = int(item_summary["Volume Logic"].iloc[0])

print(f"Bin Size : {bin_size}")

Bin Size : 24


## Step 9: Create Demand Intervals

In [10]:
max_demand = weekly_item["Weekly Demand"].max()

bins = [(0, 0)]

start = 1

while start <= max_demand:
    end = start + bin_size - 1
    bins.append((start, end))
    start += bin_size

bins

[(0, 0),
 (1, 24),
 (25, 48),
 (49, 72),
 (73, 96),
 (97, 120),
 (121, 144),
 (145, 168),
 (169, 192),
 (193, 216),
 (217, 240),
 (241, 264)]

## Step 10: Generate Frequency Distribution

In [11]:
frequency = []

for lower, upper in bins:

    if lower == 0:
        count = 0
    else:
        count = (
            weekly_item["Weekly Demand"]
            .between(lower, upper)
            .sum()
        )

    frequency.append(
        {
            "Lower": lower,
            "Upper": upper,
            "Frequency": count
        }
    )

frequency_df = pd.DataFrame(frequency)

display(frequency_df)

,Lower,Upper,Frequency
0,0,0,0
1,1,24,23
2,25,48,26
3,49,72,12
4,73,96,7
5,97,120,1
6,121,144,1
7,145,168,1
8,169,192,0
9,193,216,0


## Step 11: Include Zero-Demand Weeks

In [12]:
# TOTAL_WEEKS = 74

TOTAL_WEEKS = (
    df["Year"].astype(str) + "-" + df["Week"].astype(str)
).nunique()

print(f"Total Weeks : {TOTAL_WEEKS}")
non_zero_frequency = frequency_df.loc[1:, "Frequency"].sum()

frequency_df.loc[0, "Frequency"] = (
    TOTAL_WEEKS - non_zero_frequency
)

display(frequency_df)

Total Weeks : 74


,Lower,Upper,Frequency
0,0,0,2
1,1,24,23
2,25,48,26
3,49,72,12
4,73,96,7
5,97,120,1
6,121,144,1
7,145,168,1
8,169,192,0
9,193,216,0


## Step 12: Calculate Probability Distribution

In [13]:
frequency_df["Contribution"] = (
    frequency_df["Frequency"] / TOTAL_WEEKS
)

frequency_df["Cum Probability"] = (
    frequency_df["Contribution"].cumsum()
)

frequency_df["Mid Point"] = (
    frequency_df["Lower"] +
    frequency_df["Upper"]
) / 2

frequency_df["Weighted Sum"] = (
    frequency_df["Mid Point"] *
    frequency_df["Contribution"]
)

display(frequency_df)

,Lower,Upper,Frequency,Contribution,Cum Probability,Mid Point,Weighted Sum
0,0,0,2,0.027027,0.027027,0.0,0.000000
1,1,24,23,0.310811,0.337838,12.5,3.885135
2,25,48,26,0.351351,0.689189,36.5,12.824324
3,49,72,12,0.162162,0.851351,60.5,9.810811
4,73,96,7,0.094595,0.945946,84.5,7.993243
5,97,120,1,0.013514,0.959459,108.5,1.466216
6,121,144,1,0.013514,0.972973,132.5,1.790541
7,145,168,1,0.013514,0.986486,156.5,2.114865
8,169,192,0,0.000000,0.986486,180.5,0.000000
9,193,216,0,0.000000,0.986486,204.5,0.000000


## Step 13: Calculate Weekly Demand Statistics

In [14]:
# average_weekly_demand = frequency_df["Weighted Sum"].sum()
# print(f"Average Weekly Demand : {average_weekly_demand:.2f}")

# SERVICE_LEVEL = 0.85

# closest_row = frequency_df.iloc[
#     (frequency_df["Cum Probability"] - SERVICE_LEVEL).abs().argmin()
# ]

# d_max_week = round(closest_row["Upper"])
# average_weekly_demand = round(average_weekly_demand)

# print(f"Average Weekly Demand : {average_weekly_demand}")
# print(f"Dmax / Week : {d_max_week}")
# print(f"Selected Cum Probability : {closest_row['Cum Probability']:.4f}")

In [15]:
average_weekly_demand = round(frequency_df["Weighted Sum"].sum())

SERVICE_LEVEL = 0.85
INTERPOLATION_THRESHOLD = 0.05  # 10 percentage points

below = frequency_df[
    frequency_df["Cum Probability"] < SERVICE_LEVEL
].iloc[-1]

above = frequency_df[
    frequency_df["Cum Probability"] >= SERVICE_LEVEL
].iloc[0]

gap = above["Cum Probability"] - below["Cum Probability"]

if gap > INTERPOLATION_THRESHOLD:
    fraction = (
        (SERVICE_LEVEL - below["Cum Probability"]) / gap
    )

    d_max_week = (
        below["Upper"] +
        fraction * (above["Upper"] - below["Upper"])
    )

    d_max_week = round(d_max_week)
    method = "Interpolation"

else:
    closest_row = frequency_df.iloc[
        (frequency_df["Cum Probability"] - SERVICE_LEVEL).abs().argmin()
    ]

    d_max_week = int(closest_row["Upper"])
    method = "Nearest Bucket"

print(f"Method Used : {method}")
print(f"Dmax / Week : {d_max_week}")

Method Used : Interpolation
Dmax / Week : 72


## Step 14: Convert Weekly Demand to Monthly Demand

In [16]:
average_monthly_demand = average_weekly_demand * 4

d_max_month = d_max_week * 4

print(f"Average Monthly Demand : {average_monthly_demand:.2f}")
print(f"Dmax / Month : {d_max_month}")

Average Monthly Demand : 172.00
Dmax / Month : 288


## Step 15: Calculate Safety Stock

In [17]:
LEAD_TIME = 4

safety_stock = (
    d_max_week -
    average_weekly_demand
) * LEAD_TIME

print(f"Safety Stock : {safety_stock:.2f}")

Safety Stock : 116.00


## Step 16: Calculate Reorder Level

In [18]:
rol = (
    average_monthly_demand + safety_stock)

print(f"Reorder Level : {rol:.2f}")

Reorder Level : 288.00


## Final Result

In [19]:
result = pd.DataFrame(
    {
        "Metric": [
            "Item Code",
            "Average Weekly Demand",
            "Average Monthly Demand",
            "Maximum Weekly Demand (Dmax)",
            "Maximum Monthly Demand",
            "Safety Stock",
            "Reorder Level"
        ],
        "Value": [
            item_code,
            round(average_weekly_demand, 2),
            round(average_monthly_demand, 2),
            d_max_week,
            d_max_month,
            round(safety_stock, 2),
            round(rol, 2)
        ]
    }
)

display(result)

,Metric,Value
0,Item Code,4960.85.075.100
1,Average Weekly Demand,43
2,Average Monthly Demand,172
3,Maximum Weekly Demand (Dmax),72
4,Maximum Monthly Demand,288
5,Safety Stock,116
6,Reorder Level,288
